### LeNet 5 Pytorch Implementation

Research Paper Link :- https://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf

![LeNet5 Architecture](https://upload.wikimedia.org/wikipedia/commons/3/35/LeNet-5_architecture.svg)

In [ ]:
!pip3 install torch torchvision

### Importing Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

### Defining LeNet 5 Architecture

In [4]:
class LeNet5(nn.Module):
   def __init__(self):
      super(LeNet5,self).__init__()
      self.conv1 = nn.Conv2d(1,6,kernel_size=5,stride=1,padding=2)
      self.relu = nn.ReLU()
      self.pool = nn.AvgPool2d(kernel_size=2,stride=2)
      self.conv2 = nn.Conv2d(6,16,kernel_size=5,stride=1)
      self.fc1= nn.Linear(16*5*5,120)
      self.fc2 = nn.Linear(120,84)
      self.fc3 = nn.Linear(84,10)
      
   def forward(self,x):
      x = self.pool(self.relu(self.conv1(x)))
      x = self.pool(self.relu(self.conv2(x)))
      x = x.view(-1,16*5*5)
      x = self.relu(self.fc1(x))
      x = self.relu(self.fc2(x))
      x = self.fc3(x)
      return x

### Hyperparameters

In [5]:
batch_size = 64
epochs = 10
learning_rate = 0.001

### Data Loading and Transformation

In [6]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, transform=transform, download=True
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100.0%
100.0%
100.0%
100.0%


### Model, Loss, and Optimizer

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Model Training & Evaluation

In [ ]:
# Training Loop
train_losses, test_accuracies = [], []
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

    # Evaluation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    test_accuracies.append(accuracy)
    print(f"Accuracy on test set: {accuracy:.2f}%")

In [ ]:
# Plot Accuracy and Loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses, marker="o", linestyle="-")
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Epochs")

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), test_accuracies, marker="s", linestyle="-")
plt.xlabel("Epochs")
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Over Epochs")
plt.show()